# 🔬 Semi-Supervised Learning for Metallicity (Materials Project)
## Demonstrating SSL with limited labeled data

---

### 🎯 Objective
Train with only **1,000 labeled compositions** and compare a supervised baseline with five semi-supervised methods (Label Propagation, Label Spreading, Self-Training, Co-Training, Cluster-then-Label) and self-supervised pre-training, while the unlabeled pool grows from 1,000 to 100,000 compositions.

The full sweep (3 seeds × 4 pool sizes × 2 unlabeled-pool settings, Optuna tuning) is run by `pipeline/run_ssl_experiments.py`; this notebook loads its results and re-runs every method once live.

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
TASK = "metallicity"
PRED = ROOT / "data" / "predictions" / TASK
from IPython.display import Image, display

In [ ]:
print("=" * 100); print("STEP 1: LOADING EXPERIMENT RESULTS"); print("=" * 100)
summ = {p: json.loads((PRED / p / f'ssl_summary_{TASK}.json').read_text()) for p in ('indist', 'gnome') if (PRED / p / f'ssl_summary_{TASK}.json').exists()}
res = {p: pd.read_csv(PRED / p / f'ssl_results_{TASK}.csv') for p in summ}
for p, s in summ.items():
    print(p, json.dumps(s['training_setup'], indent=1))

In [ ]:
print("STEP 2-3: BASELINE MODEL ZOO (supervised, 1,000 labels, mean over 3 seeds)")
r = res['indist']
zoo = r[r.Method == 'Baseline'].groupby('Config')[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Composite_Score']].agg(['mean', 'std'])
zoo.sort_values(('Composite_Score', 'mean'), ascending=False).round(4)

In [ ]:
print("STEP 4: OPTUNA HYPER-PARAMETER SEARCH (25 trials, 5-fold CV)")
pd.DataFrame({'CV composite': summ['indist']['optuna_cv_scores'], 'best params': {k: str(v) for k, v in summ['indist']['tuned_params'].items()}})

In [ ]:
print("STEP 4b: SELF-SUPERVISED PRE-TRAINING (masked autoencoder on 554k GNoME)")
r[r.Method == 'Self-Supervised (AE)'].groupby('Config')[['F1-Score', 'ROC-AUC', 'Composite_Score']].agg(['mean', 'std']).round(4)

In [ ]:
print("STEP 5-8: SEMI-SUPERVISED SWEEP – mean composite per pool size")
for p, r_ in res.items():
    print(f'--- unlabeled pool: {p}')
    ssl = r_[~r_.Method.isin(['Baseline', 'Self-Supervised (AE)'])]
    display(ssl.pivot_table(index=['Method', 'Config'], columns='N_Unlabeled', values='Composite_Score', aggfunc='mean').round(4))

In [ ]:
print("STEP 9: COMPREHENSIVE RESULTS COMPARISON (best configuration per method)")
for p, s in summ.items():
    print(f'--- unlabeled pool: {p}')
    b = pd.DataFrame(s['best_per_method'])[['Method', 'Model_Config', 'Composite_Score_mean', 'Composite_Score_std', 'F1-Score_mean', 'ROC-AUC_mean', 'Accuracy_mean']]
    display(b.round(4))
    print('SSL vs best baseline: %+.2f %%' % s['ssl_improvement_over_best_baseline_pct'])

In [ ]:
print("STEP 10: LEARNING CURVES")
for p in summ:
    display(Image(str(PRED / p / f'{TASK}_learning_curves.png')))

In [ ]:
print("STEP 11: BEST MODELS COMPARISON (balanced composite metric)")
for p in summ:
    display(Image(str(PRED / p / f'{TASK}_methods_comparison_balanced.png')))

In [ ]:
print("STEP 12: LABEL-BUDGET CURVE AND FULLY-SUPERVISED UPPER BOUND")
for p in summ:
    display(Image(str(PRED / p / f'{TASK}_label_budget_curve.png')))
    print(p, 'fully supervised:', summ[p]['fully_supervised_upper_bound'])

### Live run: every method once (seed 0, 1,000 labels, 5,000 unlabeled)

In [ ]:
import ssl_core as core
from run_ssl_experiments import Preprocessor, stratified_take
from featurize import view_columns
from sklearn.model_selection import train_test_split
lab = pd.read_parquet(ROOT / 'data/processed' / f'{TASK}_features.parquet')
feat = [c for c in lab.columns if c.startswith(('MagpieData', 'frac ')) or c.endswith('-norm')]
tr, te = train_test_split(np.arange(len(lab)), test_size=0.2, stratify=lab.target, random_state=42)
prep = Preprocessor().fit(lab.iloc[tr][feat])
Xtr, ytr = prep.transform(lab.iloc[tr][feat]), lab.target.values[tr]
Xte, yte = prep.transform(lab.iloc[te][feat]), lab.target.values[te]
rng = np.random.default_rng(0)
li = stratified_take(ytr, 1000, rng)
rest = rng.permutation(np.setdiff1d(np.arange(len(ytr)), li))[:5000]
keep = stratified_take(yte, min(10000, len(yte)), rng)
d = core.Data(Xtr[li], ytr[li], Xtr[rest], Xte[keep], yte[keep])
v1n, v2n = view_columns()
v2 = [i for i, c in enumerate(prep.cols) if c in v2n or (any(k in c for k in ('Valence', 'Unfilled', 'Column', 'Row')) and (c.endswith(' mean') or c.endswith(' avg_dev')))]
v1 = [i for i in range(len(prep.cols)) if i not in v2]
best = summ['indist']['best_supervised_baseline']['config'].split(' ')[0]
params = summ['indist']['tuned_params'].get(best, {})
runs = [core.run_baseline(d, best, params=params)[0],
        core.run_label_propagation(d, 'knn', n_neighbors=10)[0],
        core.run_label_propagation(d, 'knn', n_neighbors=10, spreading=True)[0],
        core.run_self_training(d, best, threshold=0.9, max_iter=5, params=params)[0],
        core.run_co_training(d, v1, v2, best, threshold=0.85, n_rounds=8, params=params)[0],
        core.run_cluster_then_label(d, best, params=params)[0]]
pd.DataFrame(runs)[['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'Composite_Score', 'Training_Time']].round(4)

In [ ]:
print("STEP 13: DEPLOYED MODEL (selected architecture re-trained on all training labels)")
for p, s in summ.items():
    print(p, s['best_ssl_model']['Model_Config'], '->', s['deployment_model_test_metrics'])